# Consulta de Indicadores Económicos del BCCR mediante API

Este notebook permite consultar indicadores económicos del Banco Central de Costa Rica mediante el nuevo API del Sistema de Divulgación de Datos Económicos (SDDE).

El objetivo es sustituir el consumo anterior mediante Web Service XML por una consulta REST API en formato JSON, permitiendo:

- Validar la suscripción del usuario.
- Consultar un indicador económico por código.
- Solicitar rango de fechas.
- Convertir la serie diaria en corte mensual.
- Guardar los datos en una base SQLite.
- Mantener trazabilidad mediante usuario y fecha de registro.

## Librerías

- sqlite3  -> para guardar en base de datos local.
- requests -> para conectarse al API del BCCR.
- pandas   -> para transformar los datos.
- datetime -> para registrar la fecha de carga.
- Path     -> para manejar rutas del proyecto.
- getpass  -> para solicitar el token sin mostrarlo.

In [1]:
import sqlite3
import requests
import pandas as pd

from datetime import datetime
from pathlib import Path
import getpass

## Definir rutas del proyecto

In [2]:
# ============================================================
# 1. Definición de rutas del proyecto
# ============================================================

BASE_DIR = Path.cwd()

# Si el notebook se ejecuta desde la carpeta notebooks,
# se sube un nivel para quedar en la raíz del proyecto.
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

DATA_DIR = BASE_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)

DB_PATH = DATA_DIR / "bccr_indicadores.db"

print("Ruta base del proyecto:", BASE_DIR)
print("Ruta de datos:", DATA_DIR)
print("Ruta de base SQLite:", DB_PATH)

Ruta base del proyecto: C:\Users\mario\bccr-api-tipo-cambio
Ruta de datos: C:\Users\mario\bccr-api-tipo-cambio\data
Ruta de base SQLite: C:\Users\mario\bccr-api-tipo-cambio\data\bccr_indicadores.db


## Parámetros generales

In [3]:
# ============================================================
# 2. Parámetros generales
# ============================================================

BASE_URL_BCCR = "https://apim.bccr.fi.cr/SDDE/api/Bccr.GE.SDDE.Publico.Indicadores.API"

IDIOMA = "ES"

NOMBRE_TABLA = "tipo_cambio_mensual"

print("API base BCCR:", BASE_URL_BCCR)
print("Idioma:", IDIOMA)
print("Tabla destino:", NOMBRE_TABLA)

API base BCCR: https://apim.bccr.fi.cr/SDDE/api/Bccr.GE.SDDE.Publico.Indicadores.API
Idioma: ES
Tabla destino: tipo_cambio_mensual


## Solicitar correo y token

In [4]:
# ============================================================
# 3. Credenciales del API BCCR
# ============================================================

correo = input("Ingrese el correo del API BCCR: ")

# El token se solicita oculto para no dejarlo expuesto en GitHub.
token = getpass.getpass("Ingrese el token del BCCR: ")

print("Correo registrado:", correo)
print("Token cargado:", "Sí" if len(token.strip()) > 0 else "No")

Ingrese el correo del API BCCR:  mmorales@cajadeande.fi.cr
Ingrese el token del BCCR:  ········


Correo registrado: mmorales@cajadeande.fi.cr
Token cargado: Sí


## Función para validar suscripción

In [5]:
# ============================================================
# 4. Función para validar suscripción del usuario
# ============================================================

def validar_suscripcion_bccr(correo, token):
    """
    Valida la suscripción del usuario ante el API del BCCR.

    Parámetros:
    - correo: correo registrado ante el BCCR.
    - token: token asignado al usuario.

    Retorna:
    - Diccionario JSON con la respuesta del API.
    """

    url = f"{BASE_URL_BCCR}/Usuario/ValideSuscripcion"

    headers = {
        "Authorization": f"Bearer {token.strip()}",
        "Content-Type": "application/json",
        "Accept": "application/json",
        "User-Agent": "Mozilla/5.0"
    }

    payload = {
        "Correo": correo.strip(),
        "Token": token.strip()
    }

    response = requests.post(
        url,
        headers=headers,
        json=payload,
        timeout=60
    )

    print("HTTP:", response.status_code)
    print("Respuesta:", response.text[:500])

    if response.status_code != 200:
        raise Exception(
            f"Error validando suscripción. "
            f"HTTP {response.status_code}: {response.text[:500]}"
        )

    respuesta_json = response.json()

    if not respuesta_json.get("estado", False):
        raise Exception(
            f"Suscripción no válida: {respuesta_json.get('mensaje')}"
        )

    return respuesta_json

##  Ejecutar validación

In [6]:
# ============================================================
# 5. Ejecutar validación
# ============================================================

validacion = validar_suscripcion_bccr(correo, token)

validacion

HTTP: 200
Respuesta: {"estado":true,"mensaje":"Suscripción válida","datos":[]}


{'estado': True, 'mensaje': 'Suscripción válida', 'datos': []}

## Solicitar código del indicador y fechas

In [7]:
# ============================================================
# 6. Solicitar parámetros de consulta
# ============================================================

codigo_indicador = input("Ingrese el código del indicador BCCR. Ejemplo 317 para compra, 318 para venta: ")
fecha_inicio = input("Ingrese la fecha de inicio en formato yyyy/mm/dd. Ejemplo 2026/01/01: ")
fecha_fin = input("Ingrese la fecha final en formato yyyy/mm/dd. Ejemplo 2026/12/31: ")

print("Código indicador:", codigo_indicador)
print("Fecha inicio:", fecha_inicio)
print("Fecha fin:", fecha_fin)

Ingrese el código del indicador BCCR. Ejemplo 317 para compra, 318 para venta:  317
Ingrese la fecha de inicio en formato yyyy/mm/dd. Ejemplo 2026/01/01:  2020/01/01
Ingrese la fecha final en formato yyyy/mm/dd. Ejemplo 2026/12/31:  2026/05/15


Código indicador: 317
Fecha inicio: 2020/01/01
Fecha fin: 2026/05/15


## Función para consultar el indicador

In [8]:
# ============================================================
# 7. Función para consultar serie de indicador económico
# ============================================================

def consultar_serie_indicador_bccr(
    codigo_indicador,
    fecha_inicio,
    fecha_fin,
    token,
    idioma="ES"
):
    """
    Consulta una serie económica del BCCR por código de indicador.

    Parámetros:
    - codigo_indicador: código del indicador económico.
    - fecha_inicio: fecha inicial en formato yyyy/mm/dd.
    - fecha_fin: fecha final en formato yyyy/mm/dd.
    - token: token BCCR.
    - idioma: idioma de respuesta.

    Retorna:
    - Respuesta JSON del API.
    """

    url = f"{BASE_URL_BCCR}/indicadoresEconomicos/{codigo_indicador}/series"

    headers = {
        "Authorization": f"Bearer {token.strip()}",
        "Accept": "application/json",
        "User-Agent": "Mozilla/5.0"
    }

    params = {
        "fechaInicio": fecha_inicio,
        "fechaFin": fecha_fin,
        "idioma": idioma
    }

    response = requests.get(
        url,
        headers=headers,
        params=params,
        timeout=60
    )

    print("HTTP:", response.status_code)
    print("Respuesta:", response.text[:500])

    if response.status_code != 200:
        raise Exception(
            f"Error consultando indicador {codigo_indicador}. "
            f"HTTP {response.status_code}: {response.text[:500]}"
        )

    respuesta_json = response.json()

    if not respuesta_json.get("estado", False):
        raise Exception(
            f"Consulta no exitosa para indicador {codigo_indicador}: "
            f"{respuesta_json.get('mensaje')}"
        )

    return respuesta_json

## Ejecutar consulta del indicador

In [10]:
# ============================================================
# 8. Ejecutar consulta del indicador
# ============================================================

respuesta_indicador = consultar_serie_indicador_bccr(
    codigo_indicador=codigo_indicador,
    fecha_inicio=fecha_inicio,
    fecha_fin=fecha_fin,
    token=token,
    idioma=IDIOMA
)

HTTP: 200
Respuesta: {"estado":true,"mensaje":"Consulta exitosa","datos":[{"codigoIndicador":"317","nombreIndicador":"Tipo cambio compra","series":[{"fecha":"2020-01-01","valorDatoPorPeriodo":570.09000000},{"fecha":"2020-01-02","valorDatoPorPeriodo":570.09000000},{"fecha":"2020-01-03","valorDatoPorPeriodo":569.50000000},{"fecha":"2020-01-04","valorDatoPorPeriodo":569.37000000},{"fecha":"2020-01-05","valorDatoPorPeriodo":569.37000000},{"fecha":"2020-01-06","valorDatoPorPeriodo":569.37000000},{"fecha":"2020-01-07","va


## Función para convertir JSON a DataFrame diario

In [11]:
# ============================================================
# 9. Convertir respuesta JSON a DataFrame diario
# ============================================================

def convertir_json_bccr_a_dataframe(respuesta_json):
    """
    Convierte la respuesta JSON del API BCCR en un DataFrame diario.

    Retorna columnas:
    - codigo_indicador
    - nombre_indicador
    - fecha
    - valor
    """

    datos = respuesta_json.get("datos", [])

    if len(datos) == 0:
        print("La respuesta no contiene datos.")
        return pd.DataFrame()

    indicador = datos[0]

    codigo = indicador.get("codigoIndicador")
    nombre = indicador.get("nombreIndicador")
    series = indicador.get("series", [])

    if len(series) == 0:
        print("La serie no contiene registros.")
        return pd.DataFrame()

    df = pd.DataFrame(series)

    df["codigo_indicador"] = str(codigo)
    df["nombre_indicador"] = nombre
    df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
    df["valor"] = pd.to_numeric(df["valorDatoPorPeriodo"], errors="coerce")

    df = df[
        [
            "codigo_indicador",
            "nombre_indicador",
            "fecha",
            "valor"
        ]
    ]

    df = df.dropna(subset=["fecha"])
    df = df.sort_values("fecha").reset_index(drop=True)

    return df

## Crear DataFrame diario

In [12]:
# ============================================================
# 10. Crear DataFrame diario
# ============================================================

df_diario = convertir_json_bccr_a_dataframe(respuesta_indicador)

df_diario.head()

,codigo_indicador,nombre_indicador,fecha,valor
0,317,Tipo cambio compra,2020-01-01,570.09
1,317,Tipo cambio compra,2020-01-02,570.09
2,317,Tipo cambio compra,2020-01-03,569.50
3,317,Tipo cambio compra,2020-01-04,569.37
4,317,Tipo cambio compra,2020-01-05,569.37


## Función para convertir diario a mensual

In [32]:
# ============================================================
# 11. Convertir serie diaria a formato mensual
# ============================================================

def convertir_diario_a_mensual(
    df_diario,
    usuario_registro=None
):
    """
    Convierte una serie diaria en una serie mensual tomando el último dato
    disponible de cada mes.

    Retorna:
    - codigo_indicador
    - nombre_indicador
    - fecha
    - valor
    - MES_CORTE
    - USUARIO_REGISTRO
    - FECHA_REGISTRO
    """

    if usuario_registro is None:
        usuario_registro = getpass.getuser()

    if df_diario.empty:
        return pd.DataFrame()

    df = df_diario.copy()

    df["MES_CORTE"] = df["fecha"].values.astype("datetime64[M]")

    df_mensual = (
        df.sort_values("fecha")
          .groupby(["codigo_indicador", "MES_CORTE"], as_index=False)
          .tail(1)
          .copy()
    )

    df_mensual["USUARIO_REGISTRO"] = usuario_registro
    df_mensual["FECHA_REGISTRO"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    df_mensual = df_mensual[
        [
            "codigo_indicador",
            "nombre_indicador",
            "fecha",
            "valor",
            "MES_CORTE",
            "USUARIO_REGISTRO",
            "FECHA_REGISTRO"
        ]
    ]

    return df_mensual.reset_index(drop=True)

## Ejecutar conversión mensual

In [14]:
# ============================================================
# 12. Ejecutar conversión mensual
# ============================================================

nombre_columna_valor = input("Nombre de la columna de valor. Ejemplo TC_COMPRA o TC_VENTA: ")

df_mensual = convertir_diario_a_mensual(
    df_diario=df_diario,
    nombre_columna_valor=nombre_columna_valor
)

df_mensual.tail()

Nombre de la columna de valor. Ejemplo TC_COMPRA o TC_VENTA:  TC_COMPRA


,fecha,TC_COMPRA,MES_CORTE,USUARIO_REGISTRO,FECHA_REGISTRO
72,2026-01-31,492.32,2026-01-01,mario,2026-05-19 17:14:10
73,2026-02-28,466.92,2026-02-01,mario,2026-05-19 17:14:10
74,2026-03-31,462.08,2026-03-01,mario,2026-05-19 17:14:10
75,2026-04-30,452.25,2026-04-01,mario,2026-05-19 17:14:10
76,2026-05-15,451.24,2026-05-01,mario,2026-05-19 17:14:10


## Función para guardar en SQLite

In [31]:
# ============================================================
# 13. Guardar DataFrame mensual en SQLite
# ============================================================

def guardar_mensual_en_sqlite(
    df_mensual,
    db_path,
    nombre_tabla="bccr_indicadores_mensual"
):
    """
    Guarda la tabla mensual en SQLite.

    Llave primaria:
    - codigo_indicador
    - MES_CORTE

    Esto permite guardar varios indicadores en la misma tabla mensual.
    """

    if df_mensual.empty:
        print("No hay datos mensuales para guardar.")
        return pd.DataFrame()

    df = df_mensual.copy()

    df["fecha"] = pd.to_datetime(df["fecha"]).dt.strftime("%Y-%m-%d")
    df["MES_CORTE"] = pd.to_datetime(df["MES_CORTE"]).dt.strftime("%Y-%m-%d")

    with sqlite3.connect(db_path) as conn:
        cursor = conn.cursor()

        cursor.execute(f"""
            CREATE TABLE IF NOT EXISTS {nombre_tabla} (
                codigo_indicador TEXT NOT NULL,
                nombre_indicador TEXT,
                fecha TEXT NOT NULL,
                valor REAL,
                MES_CORTE TEXT NOT NULL,
                USUARIO_REGISTRO TEXT,
                FECHA_REGISTRO TEXT,
                PRIMARY KEY (codigo_indicador, MES_CORTE)
            )
        """)

        for _, row in df.iterrows():
            cursor.execute(f"""
                INSERT INTO {nombre_tabla} (
                    codigo_indicador,
                    nombre_indicador,
                    fecha,
                    valor,
                    MES_CORTE,
                    USUARIO_REGISTRO,
                    FECHA_REGISTRO
                )
                VALUES (?, ?, ?, ?, ?, ?, ?)
                ON CONFLICT(codigo_indicador, MES_CORTE)
                DO UPDATE SET
                    nombre_indicador = excluded.nombre_indicador,
                    fecha = excluded.fecha,
                    valor = excluded.valor,
                    USUARIO_REGISTRO = excluded.USUARIO_REGISTRO,
                    FECHA_REGISTRO = excluded.FECHA_REGISTRO
            """, (
                row["codigo_indicador"],
                row["nombre_indicador"],
                row["fecha"],
                row["valor"],
                row["MES_CORTE"],
                row["USUARIO_REGISTRO"],
                row["FECHA_REGISTRO"]
            ))

        conn.commit()

    print("Datos mensuales guardados correctamente.")
    print("Base:", db_path)
    print("Tabla:", nombre_tabla)
    print("Registros mensuales procesados:", len(df))

    return df

In [21]:
# ============================================================
# 10.1 Guardar DataFrame diario en SQLite
# ============================================================

def guardar_diario_en_sqlite(
    df_diario,
    db_path,
    nombre_tabla="bccr_indicadores_diario",
    usuario_registro=None
):
    """
    Guarda la serie diaria del indicador económico del BCCR en SQLite.

    Estructura final:
    - codigo_indicador
    - nombre_indicador
    - fecha
    - valor
    - MES_CORTE
    - USUARIO_REGISTRO
    - FECHA_REGISTRO

    Llave primaria:
    - codigo_indicador
    - fecha

    Si el registro ya existe, se actualiza.
    """

    if usuario_registro is None:
        usuario_registro = getpass.getuser()

    if df_diario.empty:
        print("No hay datos diarios para guardar.")
        return pd.DataFrame()

    df = df_diario.copy()

    df["MES_CORTE"] = df["fecha"].dt.to_period("M").astype(str)
    df["USUARIO_REGISTRO"] = usuario_registro
    df["FECHA_REGISTRO"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    df = df[
        [
            "codigo_indicador",
            "nombre_indicador",
            "fecha",
            "valor",
            "MES_CORTE",
            "USUARIO_REGISTRO",
            "FECHA_REGISTRO"
        ]
    ]

    df = df.dropna(subset=["fecha"])
    df = df.drop_duplicates(
        subset=["codigo_indicador", "fecha"],
        keep="last"
    )

    df["fecha"] = pd.to_datetime(df["fecha"]).dt.strftime("%Y-%m-%d")

    with sqlite3.connect(db_path) as conn:
        cursor = conn.cursor()

        cursor.execute(f"""
            CREATE TABLE IF NOT EXISTS {nombre_tabla} (
                codigo_indicador TEXT NOT NULL,
                nombre_indicador TEXT,
                fecha TEXT NOT NULL,
                valor REAL,
                MES_CORTE TEXT,
                USUARIO_REGISTRO TEXT,
                FECHA_REGISTRO TEXT,
                PRIMARY KEY (codigo_indicador, fecha)
            )
        """)

        for _, row in df.iterrows():
            cursor.execute(f"""
                INSERT INTO {nombre_tabla} (
                    codigo_indicador,
                    nombre_indicador,
                    fecha,
                    valor,
                    MES_CORTE,
                    USUARIO_REGISTRO,
                    FECHA_REGISTRO
                )
                VALUES (?, ?, ?, ?, ?, ?, ?)
                ON CONFLICT(codigo_indicador, fecha)
                DO UPDATE SET
                    nombre_indicador = excluded.nombre_indicador,
                    valor = excluded.valor,
                    MES_CORTE = excluded.MES_CORTE,
                    USUARIO_REGISTRO = excluded.USUARIO_REGISTRO,
                    FECHA_REGISTRO = excluded.FECHA_REGISTRO
            """, (
                row["codigo_indicador"],
                row["nombre_indicador"],
                row["fecha"],
                row["valor"],
                row["MES_CORTE"],
                row["USUARIO_REGISTRO"],
                row["FECHA_REGISTRO"]
            ))

        conn.commit()

    print("Datos diarios guardados correctamente.")
    print("Base:", db_path)
    print("Tabla:", nombre_tabla)
    print("Registros diarios procesados:", len(df))

    return df

## Guardar en SQLite

In [16]:
# ============================================================
# 14. Ejecutar guardado en SQLite
# ============================================================

df_guardado = guardar_mensual_en_sqlite(
    df_mensual=df_mensual,
    db_path=DB_PATH,
    nombre_tabla=NOMBRE_TABLA,
    nombre_columna_valor=nombre_columna_valor
)

df_guardado.tail()

Datos guardados correctamente.
Base: C:\Users\mario\bccr-api-tipo-cambio\data\bccr_indicadores.db
Tabla: tipo_cambio_mensual
Registros procesados: 77


,fecha,TC_COMPRA,MES_CORTE,USUARIO_REGISTRO,FECHA_REGISTRO
72,2026-01-31,492.32,2026-01-01,mario,2026-05-19 17:14:10
73,2026-02-28,466.92,2026-02-01,mario,2026-05-19 17:14:10
74,2026-03-31,462.08,2026-03-01,mario,2026-05-19 17:14:10
75,2026-04-30,452.25,2026-04-01,mario,2026-05-19 17:14:10
76,2026-05-15,451.24,2026-05-01,mario,2026-05-19 17:14:10


In [22]:
# ============================================================
# 10.2 Guardar tabla diaria
# ============================================================

df_diario_guardado = guardar_diario_en_sqlite(
    df_diario=df_diario,
    db_path=DB_PATH,
    nombre_tabla="bccr_indicadores_diario"
)

df_diario_guardado.tail()

Datos diarios guardados correctamente.
Base: C:\Users\mario\bccr-api-tipo-cambio\data\bccr_indicadores.db
Tabla: bccr_indicadores_diario
Registros diarios procesados: 2327


,codigo_indicador,nombre_indicador,fecha,valor,MES_CORTE,USUARIO_REGISTRO,FECHA_REGISTRO
2322,317,Tipo cambio compra,2026-05-11,455.24,2026-05,mario,2026-05-19 17:21:03
2323,317,Tipo cambio compra,2026-05-12,453.23,2026-05,mario,2026-05-19 17:21:03
2324,317,Tipo cambio compra,2026-05-13,452.75,2026-05,mario,2026-05-19 17:21:03
2325,317,Tipo cambio compra,2026-05-14,451.48,2026-05,mario,2026-05-19 17:21:03
2326,317,Tipo cambio compra,2026-05-15,451.24,2026-05,mario,2026-05-19 17:21:03


## Consultar tabla guardada

In [17]:
# ============================================================
# 15. Consultar datos guardados
# ============================================================

with sqlite3.connect(DB_PATH) as conn:
    df_sql = pd.read_sql_query(
        f"""
        SELECT *
        FROM {NOMBRE_TABLA}
        ORDER BY fecha
        """,
        conn,
        parse_dates=["fecha", "MES_CORTE", "FECHA_REGISTRO"]
    )

df_sql.tail()

,fecha,TC_COMPRA,MES_CORTE,USUARIO_REGISTRO,FECHA_REGISTRO
72,2026-01-31,492.32,2026-01-01,mario,2026-05-19 17:14:10
73,2026-02-28,466.92,2026-02-01,mario,2026-05-19 17:14:10
74,2026-03-31,462.08,2026-03-01,mario,2026-05-19 17:14:10
75,2026-04-30,452.25,2026-04-01,mario,2026-05-19 17:14:10
76,2026-05-15,451.24,2026-05-01,mario,2026-05-19 17:14:10


## Resumen de validación

In [18]:
# ============================================================
# 16. Resumen de validación
# ============================================================

resumen = {
    "tabla": NOMBRE_TABLA,
    "cantidad_registros": len(df_sql),
    "fecha_minima": df_sql["fecha"].min(),
    "fecha_maxima": df_sql["fecha"].max(),
    "ultima_fecha_registro": df_sql["FECHA_REGISTRO"].max()
}

resumen

{'tabla': 'tipo_cambio_mensual',
 'cantidad_registros': 77,
 'fecha_minima': Timestamp('2020-01-31 00:00:00'),
 'fecha_maxima': Timestamp('2026-05-15 00:00:00'),
 'ultima_fecha_registro': Timestamp('2026-05-19 17:14:10')}

## Función integrada

In [33]:
# ============================================================
# 17. Función integrada de consulta y guardado diario y mensual
# ============================================================

def ejecutar_proceso_indicador_bccr(
    codigo_indicador,
    fecha_inicio,
    fecha_fin,
    token,
    db_path,
    nombre_tabla_diaria="bccr_indicadores_diario",
    nombre_tabla_mensual="bccr_indicadores_mensual",
    idioma="ES"
):
    """
    Ejecuta el proceso completo:

    1. Consulta indicador BCCR.
    2. Convierte JSON a DataFrame diario.
    3. Guarda tabla diaria en SQLite.
    4. Convierte diario a mensual.
    5. Guarda tabla mensual en SQLite.
    """

    respuesta = consultar_serie_indicador_bccr(
        codigo_indicador=codigo_indicador,
        fecha_inicio=fecha_inicio,
        fecha_fin=fecha_fin,
        token=token,
        idioma=idioma
    )

    df_diario = convertir_json_bccr_a_dataframe(respuesta)

    df_diario_guardado = guardar_diario_en_sqlite(
        df_diario=df_diario,
        db_path=db_path,
        nombre_tabla=nombre_tabla_diaria
    )

    df_mensual = convertir_diario_a_mensual(
        df_diario=df_diario
    )

    df_mensual_guardado = guardar_mensual_en_sqlite(
        df_mensual=df_mensual,
        db_path=db_path,
        nombre_tabla=nombre_tabla_mensual
    )

    return df_diario_guardado, df_mensual_guardado

In [34]:
df_diario_resultado, df_mensual_resultado = ejecutar_proceso_indicador_bccr(
    codigo_indicador=codigo_indicador,
    fecha_inicio=fecha_inicio,
    fecha_fin=fecha_fin,
    token=token,
    db_path=DB_PATH,
    nombre_tabla_diaria="bccr_indicadores_diario",
    nombre_tabla_mensual="bccr_indicadores_mensual",
    idioma=IDIOMA
)

df_mensual_resultado.tail()

HTTP: 200
Respuesta: {"estado":true,"mensaje":"Consulta exitosa","datos":[{"codigoIndicador":"317","nombreIndicador":"Tipo cambio compra","series":[{"fecha":"2020-01-01","valorDatoPorPeriodo":570.09000000},{"fecha":"2020-01-02","valorDatoPorPeriodo":570.09000000},{"fecha":"2020-01-03","valorDatoPorPeriodo":569.50000000},{"fecha":"2020-01-04","valorDatoPorPeriodo":569.37000000},{"fecha":"2020-01-05","valorDatoPorPeriodo":569.37000000},{"fecha":"2020-01-06","valorDatoPorPeriodo":569.37000000},{"fecha":"2020-01-07","va
Datos diarios guardados correctamente.
Base: C:\Users\mario\bccr-api-tipo-cambio\data\bccr_indicadores.db
Tabla: bccr_indicadores_diario
Registros diarios procesados: 2332
Datos mensuales guardados correctamente.
Base: C:\Users\mario\bccr-api-tipo-cambio\data\bccr_indicadores.db
Tabla: bccr_indicadores_mensual
Registros mensuales procesados: 77


,codigo_indicador,nombre_indicador,fecha,valor,MES_CORTE,USUARIO_REGISTRO,FECHA_REGISTRO
72,317,Tipo cambio compra,2026-01-31,492.32,2026-01-01,mario,2026-05-19 17:28:58
73,317,Tipo cambio compra,2026-02-28,466.92,2026-02-01,mario,2026-05-19 17:28:58
74,317,Tipo cambio compra,2026-03-31,462.08,2026-03-01,mario,2026-05-19 17:28:58
75,317,Tipo cambio compra,2026-04-30,452.25,2026-04-01,mario,2026-05-19 17:28:58
76,317,Tipo cambio compra,2026-05-20,449.66,2026-05-01,mario,2026-05-19 17:28:58


In [36]:
with sqlite3.connect(DB_PATH) as conn:
    resumen_diario = pd.read_sql_query(
        """
        SELECT 
            codigo_indicador,
            nombre_indicador,
            MIN(fecha) AS fecha_minima,
            MAX(fecha) AS fecha_maxima,
            COUNT(*) AS cantidad_registros
        FROM bccr_indicadores_diario
        GROUP BY codigo_indicador, nombre_indicador
        ORDER BY codigo_indicador
        """,
        conn
    )

resumen_diario

,codigo_indicador,nombre_indicador,fecha_minima,fecha_maxima,cantidad_registros
0,317,Tipo cambio compra,2020-01-01,2026-05-20,2332


In [35]:
with sqlite3.connect(DB_PATH) as conn:
    df_sql_mensual = pd.read_sql_query(
        """
        SELECT *
        FROM bccr_indicadores_mensual
        ORDER BY codigo_indicador, MES_CORTE
        """,
        conn,
        parse_dates=["fecha", "MES_CORTE", "FECHA_REGISTRO"]
    )

df_sql_mensual.tail()

,codigo_indicador,nombre_indicador,fecha,valor,MES_CORTE,USUARIO_REGISTRO,FECHA_REGISTRO
72,317,Tipo cambio compra,2026-01-31,492.32,2026-01-01,mario,2026-05-19 17:28:58
73,317,Tipo cambio compra,2026-02-28,466.92,2026-02-01,mario,2026-05-19 17:28:58
74,317,Tipo cambio compra,2026-03-31,462.08,2026-03-01,mario,2026-05-19 17:28:58
75,317,Tipo cambio compra,2026-04-30,452.25,2026-04-01,mario,2026-05-19 17:28:58
76,317,Tipo cambio compra,2026-05-20,449.66,2026-05-01,mario,2026-05-19 17:28:58


In [30]:
with sqlite3.connect(DB_PATH) as conn:
    df_sql_mensual = pd.read_sql_query(
        """
        SELECT *
        FROM tipo_cambio_mensual
        ORDER BY fecha
        """,
        conn,
        parse_dates=["fecha", "MES_CORTE", "FECHA_REGISTRO"]
    )

df_sql_mensual.tail()

,fecha,TC_COMPRA,MES_CORTE,USUARIO_REGISTRO,FECHA_REGISTRO
73,2026-02-28,466.92,2026-02-01,mario,2026-05-19 17:23:44
74,2026-03-31,462.08,2026-03-01,mario,2026-05-19 17:23:44
75,2026-04-30,452.25,2026-04-01,mario,2026-05-19 17:23:44
76,2026-05-15,451.24,2026-05-01,mario,2026-05-19 17:22:38
77,2026-05-20,449.66,2026-05-01,mario,2026-05-19 17:23:44


In [27]:
with sqlite3.connect(DB_PATH) as conn:
    df_sql_diario = pd.read_sql_query(
        """
        SELECT *
        FROM bccr_indicadores_diario
        ORDER BY codigo_indicador, fecha
        """,
        conn,
        parse_dates=["fecha", "FECHA_REGISTRO"]
    )

df_sql_diario.tail()

,codigo_indicador,nombre_indicador,fecha,valor,MES_CORTE,USUARIO_REGISTRO,FECHA_REGISTRO
2327,317,Tipo cambio compra,2026-05-16,449.17,2026-05,mario,2026-05-19 17:23:44
2328,317,Tipo cambio compra,2026-05-17,449.17,2026-05,mario,2026-05-19 17:23:44
2329,317,Tipo cambio compra,2026-05-18,449.17,2026-05,mario,2026-05-19 17:23:44
2330,317,Tipo cambio compra,2026-05-19,449.52,2026-05,mario,2026-05-19 17:23:44
2331,317,Tipo cambio compra,2026-05-20,449.66,2026-05,mario,2026-05-19 17:23:44
